# BrainScaleS-2 primitive noise collection

This notebook only configures and invokes the canonical CLI. It does not duplicate collection or fitting logic. Use the `EBRAINS-experimental` kernel.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path('/mnt/user/shared/AnalogAttention')
SPIKING_CALIBRATION_PATH = None  # Path('/path/to/spiking_calibration.pbin')
HAGEN_CALIBRATION_PATH = None  # Path('/path/to/hagen_calibration.pbin')
RUN_PROBE = True
RUN_QUICK_SMOKE = False
RUN_FULL_COLLECTION = False
RUN_VALIDATE_EXISTING = False
EXISTING_OUTPUT_DIR = None
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'brainscales2-primitives' / RUN_ID
CLI = REPO_ROOT / 'scripts' / 'evaluation' / 'brainscales2_primitive_noise.py'
assert CLI.is_file(), CLI

In [ ]:
def run_cli(*arguments):
    command = [sys.executable, str(CLI), *(str(value) for value in arguments)]
    print(' '.join(command), flush=True)
    subprocess.run(command, cwd=REPO_ROOT, check=True)

def calibration_args():
    if SPIKING_CALIBRATION_PATH is None or HAGEN_CALIBRATION_PATH is None:
        raise RuntimeError('Set explicit spiking and Hagen .pbin paths')
    return [
        '--spiking-calibration', Path(SPIKING_CALIBRATION_PATH).expanduser().resolve(),
        '--hagen-calibration', Path(HAGEN_CALIBRATION_PATH).expanduser().resolve(),
    ]

In [ ]:
if RUN_PROBE:
    run_cli('--phase', 'probe', '--backend', 'hardware', '--output-dir', OUTPUT_DIR)
    print((OUTPUT_DIR / 'probe.json').read_text())

In [ ]:
if RUN_QUICK_SMOKE:
    run_cli(
        '--phase', 'all', '--primitive', 'all', '--backend', 'hardware', '--quick',
        '--output-dir', OUTPUT_DIR / 'quick', *calibration_args(),
    )

In [ ]:
if RUN_FULL_COLLECTION:
    run_cli(
        '--phase', 'all', '--primitive', 'all', '--backend', 'hardware',
        '--output-dir', OUTPUT_DIR / 'full', *calibration_args(),
    )

In [ ]:
if RUN_VALIDATE_EXISTING:
    if EXISTING_OUTPUT_DIR is None:
        raise RuntimeError('Set EXISTING_OUTPUT_DIR')
    run_cli(
        '--phase', 'validate', '--backend', 'hardware',
        '--output-dir', Path(EXISTING_OUTPUT_DIR).expanduser().resolve(),
        *calibration_args(),
    )

result_dir = (OUTPUT_DIR / 'full') if (OUTPUT_DIR / 'full').is_dir() else OUTPUT_DIR / 'quick'
calibration = result_dir / 'primitive_noise_calibration.json'
if calibration.is_file():
    display(json.loads(calibration.read_text()))